# 12 · Publication figures and LaTeX tables (v3)
**Purpose:** render every manuscript figure and table **only** from the CSVs written by 04 and 09–11 and 13 (no recomputation, so figures always match the statistics).

**Design rules (journal quality):**
- vector PDF (fonts embedded as TrueType, `pdf.fonttype 42`) + 600-dpi PNG; sizes fit a 6.5-in text width (full) or 3.2 in (half);
- one sans font, 7–8 pt, hairline axes, no top/right spines, panel letters **a, b, c**;
- colour carries one job per figure: **detector type** (categorical: fine-tuned / shallow / zero-shot — validated colour-blind-safe set, plus a distinct marker shape per type so print in greyscale still works), **magnitude** (one-hue blue ramp), or **signed change** (blue ↔ red with a neutral grey midpoint);
- every figure writes its exact plotted numbers to `figures/source_data/<figure>.csv` (table-view twin, required by many Q1 journals) and is listed in `figures/figure_manifest.csv`.

**v3 (after the post-hoc checks of notebook 13):** Fig. 3b and 4a show **seed + document** CIs when `09_statistics/posthoc/` exists; Fig. 9 uses the **5 %** source-validation threshold; new Figs. 11–13 and Tables 7–8. If notebook 13 has not been run, these blocks are skipped (and Fig. 9 falls back to the 1 % threshold from 09).

| Figure | Question | Source CSV |
|---|---|---|
| 1 Main AUROC overview | how good is each detector in each language? | `08_results/metrics_agg.csv` |
| 2 Cross-lingual transfer matrices | RQ2 | `09_statistics/rq2_transfer_matrix.csv` |
| 3 Family vs script | RQ2 | `rq2_transfer_matrix.csv`, `rq2_family_vs_script.csv`, `posthoc/a2_*` |
| 4 Multilingual vs monolingual, leave-one-language-out | RQ1/RQ2 | `rq1_multi_vs_mono.csv`, `rq2_lolo.csv`, `posthoc/a1_*` |
| 5 Unseen generator families | RQ3 | `rq3_lofo.csv` |
| 6 Robustness heat-map | RQ4 | `rq4_robustness.csv` |
| 7 Homoglyph attack and Unicode defence | RQ4 | `rq4_defence.csv` |
| 8 External COLING-2025 Kazakh test | RQ3 | `external_coling_kk.csv` |
| 9 Operating-point transfer (threshold at 5 % FPR) | deployment | `posthoc/a4_threshold_transfer.csv` |
| 10 AUROC by generator | error analysis | `08_results/error_generator_prompt.csv` |
| 11 Why the Kazakh-adapted scorer fails | RQ5 mechanism | `rq5_*.csv`, `posthoc/b_*` |
| 12 Llama-3.3: unseen or atypical? | RQ3 | `posthoc/c_*` |
| 13 Score fusion | complementarity | `posthoc/d_ensemble.csv` |
| A1 Kazakh-adapted vs generic scorer | RQ5 | `rq5_language_adapted_scorer.csv` |
| A2 Length tertiles | shortcut check | `error_length.csv` |
| A3 Error overlap | complementarity | `error_overlap_jaccard.csv` |
| A4 Dataset shortcut audit | data quality | `08_results/data_audit.csv` |

**Tables:** 1 main AUROC · 2 low-FPR TPR (with CIs) · 3 family vs script · 4 LOFO · 5 robustness · 6 external · 7 RQ1 seed-aware · 8 RQ2 seed-aware.

**Outputs:** `10_figures_tables/figures/*.pdf|png`, `figures/source_data/*.csv`, `10_figures_tables/tables/*.tex`. **Runtime:** CPU, < 2 min.

In [ ]:
# ==== Bootstrap (identical in every notebook) ====
NOTEBOOK = "12_figures_tables"
PROJECT_ROOT = "/content/drive/MyDrive/multilingual-ai-text-detection"   # <- edit if needed
import os, json
try:
    from google.colab import drive; drive.mount("/content/drive")
except ImportError:
    pass
os.environ["MATD_ROOT"] = PROJECT_ROOT
def load_common(path=f"{PROJECT_ROOT}/notebooks/00_common.ipynb"):
    nb = json.load(open(path, encoding="utf-8"))
    code = "\n\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code")
    exec(compile(code, path, "exec"), globals())
load_common()
log = get_logger(NOTEBOOK)

## Style, palette and helpers

In [ ]:
import matplotlib
IN_NB = "ipykernel" in sys.modules
if not IN_NB:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
import traceback

# ---- ink & chrome (light print surface) ----
INK, INK2, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
# ---- categorical: detector type (validated: CVD ΔE ≥ 9.2 all pairs; aqua < 3:1 → always paired with marker shape + legend) ----
TYPE_COLOR = {"fine-tuned": "#2a78d6", "shallow": "#eb6834", "zero-shot": "#1baf7a"}
TYPE_MARK = {"fine-tuned": "o", "shallow": "s", "zero-shot": "^"}
# ---- sequential (magnitude): one-hue blue ramp 100→700 ----
BLUE = ["#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec", "#5598e7", "#3987e5", "#2a78d6", "#256abf", "#1c5cab", "#184f95", "#104281", "#0d366b"]
SEQ = LinearSegmentedColormap.from_list("seq_blue", ["#f4f8fe"] + BLUE)
SEQ_ORANGE = LinearSegmentedColormap.from_list("seq_orange", ["#fdf3ee", "#f6c3ac", "#eb6834", "#b8461b", "#7a2c0e"])
# ---- diverging (signed change): blue ↔ red, neutral grey midpoint, equal arms ----
DIV = LinearSegmentedColormap.from_list("div", ["#0d366b", "#3987e5", "#cde2fb", "#f0efec", "#f6c9c8", "#e34948", "#8f1f1f"])

FULL, HALF = 6.5, 3.2          # inches; adjust to the venue's text width
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 7.5, "axes.titlesize": 8, "axes.labelsize": 7.5,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7, "legend.frameon": False,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.6, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": INK2, "ytick.color": INK2, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": False, "grid.color": GRID, "grid.linewidth": 0.5, "lines.linewidth": 1.2,
    "figure.dpi": 150, "savefig.dpi": 600, "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
    "pdf.fonttype": 42, "ps.fonttype": 42, "axes.titleweight": "bold", "axes.titlelocation": "left"})

ORDER = ["kk", "ky", "uz", "tr", "ru", "en"]          # Turkic-Cyrillic, Turkic-Latin, IE-Cyrillic, IE-Latin
LNAME = {l: LANG_META[l]["name"] for l in ORDER}
SCR = {l: LANG_META[l]["script"][:3] + "." for l in ORDER}
DET_NAME = {"mdeberta": "mDeBERTa-v3", "xlmr": "XLM-R", "ngram": "Char n-gram LR", "stylo": "Stylometric LR",
            "binoculars-generic": "Binoculars", "binoculars-kazakh": "Binoculars (kk LM)",
            "fastdetect-generic": "Fast-DetectGPT", "fastdetect-kazakh": "Fast-DetectGPT (kk LM)",
            "loglik-generic": "Log-likelihood", "loglik-kazakh": "Log-likelihood (kk LM)",
            "logrank-generic": "Log-rank", "logrank-kazakh": "Log-rank (kk LM)"}
DET_TYPE = {"mdeberta": "fine-tuned", "xlmr": "fine-tuned", "ngram": "shallow", "stylo": "shallow"}
DET_TYPE |= {d: "zero-shot" for d in DET_NAME if d not in DET_TYPE}
DET_ORDER = list(DET_NAME)
FT_SHALLOW = ["mdeberta", "xlmr", "ngram", "stylo"]
DET_MARK = {"mdeberta": "o", "xlmr": "D", "ngram": "s", "stylo": "^"}
GEN_NAME = {"gpt41mini": "GPT-4.1-mini", "haiku45": "Claude Haiku 4.5", "gem25lite": "Gemini 2.5 Flash-Lite",
            "llama33_70b": "Llama-3.3-70B", "dsv3": "DeepSeek-V3"}
FAM_NAME = {"openai": "OpenAI\n(GPT-4.1-mini)", "anthropic": "Anthropic\n(Claude Haiku 4.5)", "google": "Google\n(Gemini 2.5 FL)",
            "meta": "Meta\n(Llama-3.3-70B)", "deepseek": "DeepSeek\n(DeepSeek-V3)"}
FAM_ORDER = ["openai", "anthropic", "google", "meta", "deepseek"]

S, R, F, T = P["stats"], P["results"], P["figures"], P["tables"]
PH = S / "posthoc"                          # notebook 13 outputs (optional)
SRC = F / "source_data"; SRC.mkdir(parents=True, exist_ok=True)
MANIFEST = []

def save(fig, name, data, sources, caption):
    """Save PDF + PNG, the plotted numbers as CSV, and a manifest row."""
    for ext in ("pdf", "png"):
        fig.savefig(F / f"{name}.{ext}")
    data.to_csv(SRC / f"{name}.csv", index=False)
    MANIFEST.append({"figure": name, "sources": ";".join(sources), "caption_draft": caption})
    plt.close(fig); log.info(f"saved {name}")
    if IN_NB:                                   # preview in Colab
        from IPython.display import Image, display
        display(Image(filename=str(F / f"{name}.png"), width=900))

def panel(ax, letter, x=-0.02, y=1.04):
    ax.text(x, y, letter, transform=ax.transAxes, fontsize=9, fontweight="bold", va="bottom", ha="right")

def hairgrid(ax, axis="x"):
    ax.grid(True, axis=axis, color=GRID, lw=0.5, zorder=0); ax.set_axisbelow(True)

def zero_line(ax, orient="v"):
    (ax.axvline if orient == "v" else ax.axhline)(0, color=INK2, lw=0.6, zorder=1)

def lang_tick(l):
    return f"{l}\n{SCR[l]}"

def heat(ax, M, cmap, vmin, vmax, fmt="{:.2f}", fs=6, mask=None, dim=None):
    """Annotated heat-map; text colour switches on cell luminance; `dim` (bool array) greys out uncertain cells."""
    im = ax.imshow(M, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
    norm = plt.Normalize(vmin, vmax)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M[i, j]
            if np.isnan(v) or (mask is not None and mask[i, j]):
                continue
            r, g, b, _ = cmap(norm(v)); lum = 0.2126 * r + 0.7152 * g + 0.0722 * b
            col = "white" if lum < 0.5 else INK
            if dim is not None and dim[i, j]:
                col = MUTED if lum >= 0.5 else "#d9d9d9"
            txt = fmt.format(v)
            if txt.lstrip("+-") == "0.00":
                txt = "0.00"
            ax.text(j, i, txt, ha="center", va="center", fontsize=fs, color=col)
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)
    return im

def type_legend(ax_or_fig, types=("fine-tuned", "shallow", "zero-shot"), **kw):
    h = [Line2D([], [], ls="", marker=TYPE_MARK[t], color=TYPE_COLOR[t], mec="white", mew=0.5, ms=5, label=t) for t in types]
    return ax_or_fig.legend(handles=h, **kw)

def run(fn):
    """Run one figure/table block; a missing input never stops the others."""
    try:
        fn()
    except FileNotFoundError as e:
        log.warning(f"{fn.__name__}: skipped — missing input ({e.filename})")
    except Exception:
        log.error(f"{fn.__name__} failed:\n{traceback.format_exc()}")

agg = pd.read_csv(R / "metrics_agg.csv")
MAIN = agg[(agg["eval"] == "test") & (((agg.exp == "E2") & (agg.train == "all") & (agg.heldout == "none")) | (agg.exp == "E5"))]

## Fig. 1 — Main results: test AUROC of every detector in every language
Fine-tuned and shallow detectors are trained on all six languages (E2/all, mean of seeds); zero-shot detectors need no training (E5). Error bars: 95 % hierarchical bootstrap CI.

In [ ]:
def fig1_main_auroc():
    d = MAIN[MAIN.lang.isin(ORDER)].copy()
    dets = [x for x in DET_ORDER if x in set(d.detector)]
    y = {det: i for i, det in enumerate(dets)}
    fig, axes = plt.subplots(1, 6, figsize=(FULL, 0.19 * len(dets) + 0.9), sharey=True, gridspec_kw={"wspace": 0.12})
    lo = max(0.4, np.floor((d.ci_lo.min() - 0.02) * 20) / 20)
    for ax, l in zip(axes, ORDER):
        g = d[d.lang == l].set_index("detector").reindex(dets)
        hairgrid(ax)
        for det in dets:
            r = g.loc[det]; t = DET_TYPE[det]
            ax.errorbar(r.auroc, y[det], xerr=[[r.auroc - r.ci_lo], [r.ci_hi - r.auroc]], fmt=TYPE_MARK[t], ms=3.6,
                        color=TYPE_COLOR[t], mec="white", mew=0.4, elinewidth=0.8, capsize=0, zorder=3)
        for b in (1.5, 3.5):
            ax.axhline(b, color=GRID, lw=0.6)
        ax.set_xlim(lo, 1.005); ax.set_xticks([x for x in (0.4, 0.6, 0.8, 1.0) if x >= lo])
        ax.set_xticks([x for x in np.arange(0.4, 1.0, 0.1) if x >= lo], minor=True)
        ax.grid(True, axis="x", which="minor", color=GRID, lw=0.3); ax.tick_params(axis="x", which="minor", length=0)
        ax.set_xticklabels([f"{x:.1f}" for x in ax.get_xticks()])
        ax.set_title(f"{l}  {LNAME[l]}", fontsize=7.5, loc="left", pad=9)
        fam = "Turkic" if LANG_META[l]["family"] == "Turkic" else "Indo-Eur."
        ax.text(0, 1.0, f"{fam} · {SCR[l]}", transform=ax.transAxes, fontsize=6, color=MUTED, va="bottom")
    axes[0].set_yticks(range(len(dets)), [DET_NAME[x] for x in dets]); axes[0].invert_yaxis()
    fig.supxlabel("Test AUROC (95% CI)", fontsize=7.5, y=0.01)
    type_legend(fig, loc="upper center", ncol=3, bbox_to_anchor=(0.55, 1.06), handletextpad=0.3, columnspacing=1.2)
    save(fig, "fig1_main_auroc", d[["detector", "lang", "auroc", "ci_lo", "ci_hi", "n_seeds"]], ["08_results/metrics_agg.csv"],
         "Test AUROC per language. Fine-tuned/shallow detectors trained on all six languages; zero-shot detectors untrained.")
run(fig1_main_auroc)

## Fig. 2 — Cross-lingual transfer matrices (RQ2)
Rows: training language (monolingual E1 model); columns: test language. White rules separate Turkic from Indo-European languages; the diagonal (in-language) is outlined.

In [ ]:
def fig2_transfer():
    tm = pd.read_csv(S / "rq2_transfer_matrix.csv")
    dets = [x for x in FT_SHALLOW if x in set(tm.detector)]
    nr = int(np.ceil(len(dets) / 2))
    fig, axes = plt.subplots(nr, 2, figsize=(FULL, 2.9 * nr), squeeze=False, gridspec_kw={"wspace": 0.3, "hspace": 0.38})
    for k, (ax, det) in enumerate(zip(axes.flat, dets)):
        M = tm[tm.detector == det].pivot(index="src", columns="tgt", values="auroc").reindex(index=ORDER, columns=ORDER).values
        im = heat(ax, M, SEQ, 0.5, 1.0, fmt="{:.2f}", fs=6.2)
        for i in range(6):
            ax.add_patch(Rectangle((i - 0.5, i - 0.5), 1, 1, fill=False, ec=INK, lw=0.9))
        ax.axhline(3.5, color="white", lw=2); ax.axvline(3.5, color="white", lw=2)
        ax.set_xticks(range(6), [lang_tick(l) for l in ORDER]); ax.set_yticks(range(6), [lang_tick(l).replace("\n", " ") for l in ORDER])
        ax.set_xlabel("test language", fontsize=7, color=INK2)
        if k % 2 == 0:
            ax.set_ylabel("training language", fontsize=7, color=INK2)
        off = M[~np.eye(6, dtype=bool)]
        ax.set_title(DET_NAME[det], fontsize=8, pad=4)
        ax.text(1.0, 1.01, f"mean off-diag. {np.nanmean(off):.3f}", transform=ax.transAxes, ha="right", va="bottom", fontsize=6.2, color=INK2)
        panel(ax, "abcd"[k], x=-0.17, y=1.0)
    for ax in axes.flat[len(dets):]:
        ax.axis("off")
    cb = fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.55, pad=0.02, aspect=30)
    cb.set_label("AUROC", fontsize=7); cb.outline.set_visible(False); cb.ax.tick_params(labelsize=6.5, length=2)
    save(fig, "fig2_transfer_matrices", tm, ["09_statistics/rq2_transfer_matrix.csv"],
         "Cross-lingual transfer: AUROC of monolingual detectors (rows) on each test language (columns).")
run(fig2_transfer)

## Fig. 3 — Does transfer follow language family or script? (RQ2)
**a** Off-diagonal transfer AUROC grouped by the relation between training and test language (each dot = one ordered language pair; bar = median).
**b** Fixed-effects coefficients (logit AUROC ~ same script + same family + source + target): 95 % CI from the **seed + document** bootstrap of notebook 13 when available (otherwise document bootstrap); exact permutation p (coarse: 6 languages).

In [ ]:
def rel(s, t):
    f = LANG_META[s]["family"] == LANG_META[t]["family"]; c = LANG_META[s]["script"] == LANG_META[t]["script"]
    return {(True, True): "same family\n& script", (True, False): "same family\nonly", (False, True): "same script\nonly", (False, False): "neither"}[(f, c)]
REL_ORDER = ["same family\n& script", "same family\nonly", "same script\nonly", "neither"]
REL_SHORT = ["family\n+ script", "family\nonly", "script\nonly", "neither"]

def fig3_family_script():
    tm = pd.read_csv(S / "rq2_transfer_matrix.csv"); tm = tm[tm.src != tm.tgt].copy()
    tm["relation"] = [rel(s, t) for s, t in zip(tm.src, tm.tgt)]
    cf = pd.read_csv(S / "rq2_family_vs_script.csv"); ci_kind = "document bootstrap"
    if (PH / "a2_rq2_seedaware.csv").exists():           # seed + document CIs (notebook 13)
        a2 = pd.read_csv(PH / "a2_rq2_seedaware.csv").set_index("detector")
        for coef in ("family", "script"):
            for b in ("lo", "hi"):
                cf[f"ci_{b}_{coef}"] = [a2.loc[d, f"ci_seed_doc_{coef}_{b}"] if d in a2.index else v
                                        for d, v in zip(cf.detector, cf[f"ci_{b}_{coef}"])]
        ci_kind = "seed + document bootstrap"
    dets = [x for x in FT_SHALLOW if x in set(tm.detector)]
    fig = plt.figure(figsize=(FULL, 4.1))
    gs = fig.add_gridspec(2, len(dets), height_ratios=[1.35, 1], hspace=0.62, wspace=0.12)
    rng = np.random.default_rng(0)
    ax0 = None
    for k, det in enumerate(dets):
        ax = fig.add_subplot(gs[0, k], sharey=ax0); ax0 = ax0 or ax
        g = tm[tm.detector == det]; t = DET_TYPE[det]
        hairgrid(ax, "y")
        for i, r_ in enumerate(REL_ORDER):
            v = g[g.relation == r_].auroc.values
            ax.scatter(i + rng.uniform(-0.17, 0.17, len(v)), v, s=11, marker=TYPE_MARK[t], color=TYPE_COLOR[t], ec="white", lw=0.4, zorder=3)
            if len(v):
                ax.plot([i - 0.28, i + 0.28], [np.median(v)] * 2, color=INK, lw=1.3, zorder=4)
        ax.set_xticks(range(4), REL_SHORT, fontsize=6); ax.set_title(DET_NAME[det], fontsize=7.5)
        if k:
            plt.setp(ax.get_yticklabels(), visible=False)
        else:
            ax.set_ylabel("off-diagonal AUROC"); panel(ax, "a", x=-0.18)
    # b — coefficients
    ax = fig.add_subplot(gs[1, :]); hairgrid(ax)
    cf = cf.set_index("detector").reindex(dets)
    for j, (coef, lab, col, mk, dy) in enumerate([("family", "same family", TYPE_COLOR["fine-tuned"], "o", -0.13),
                                                   ("script", "same script", TYPE_COLOR["shallow"], "s", 0.13)]):
        for i, det in enumerate(dets):
            r_ = cf.loc[det]; v = r_[f"coef_same_{coef}"]
            ax.errorbar(v, i + dy, xerr=[[v - r_[f"ci_lo_{coef}"]], [r_[f"ci_hi_{coef}"] - v]], fmt=mk, ms=4, color=col, mec="white",
                        mew=0.4, elinewidth=0.9, capsize=0, zorder=3, label=lab if i == 0 else None)
            ax.text(r_[f"ci_hi_{coef}"] + 0.02, i + dy, f"perm. p={r_[f'perm_p_{coef}']:.2f}", va="center", fontsize=5.8, color=INK2)
    zero_line(ax); ax.set_yticks(range(len(dets)), [DET_NAME[x] for x in dets]); ax.invert_yaxis()
    ax.set_xlabel(f"effect on logit(AUROC)  (95% CI, {ci_kind})")
    ax.legend(loc="lower left", bbox_to_anchor=(0.62, 1.0), ncol=2, handletextpad=0.3, borderaxespad=0)
    xmax = np.nanmax(cf[["ci_hi_family", "ci_hi_script"]].values); ax.set_xlim(min(-0.2, np.nanmin(cf[["ci_lo_family", "ci_lo_script"]].values) - 0.05), xmax + 0.35)
    panel(ax, "b", x=-0.005, y=1.05)
    out = pd.concat([tm.assign(panel="a"), cf.reset_index().assign(panel="b")], ignore_index=True)
    save(fig, "fig3_family_vs_script", out, ["09_statistics/rq2_transfer_matrix.csv", "09_statistics/rq2_family_vs_script.csv"],
         "Transfer by language relation (a) and fixed-effects family/script coefficients (b).")
run(fig3_family_script)

## Fig. 4 — Value of target-language data (RQ1, RQ2)
**a** ΔAUROC = multilingual (E2/all) − monolingual (E1, target only), 95 % CI from the seed + document bootstrap (notebook 13) when available. **b** ΔAUROC = leave-target-out (E2/no-L) − multilingual (E2/all). Positive in **a** = adding other languages helps; negative in **b** = cost of never seeing the target language.

In [ ]:
def fig4_rq1_lolo():
    r1 = pd.read_csv(S / "rq1_multi_vs_mono.csv"); lo = pd.read_csv(S / "rq2_lolo.csv")
    if (PH / "a1_rq1_seedaware.csv").exists():           # replace document-only CIs by seed + document CIs
        a1 = pd.read_csv(PH / "a1_rq1_seedaware.csv").set_index(["detector", "target"])
        key = list(zip(r1.detector, r1.target))
        r1["ci_lo"] = [a1.loc[k, "ci_lo_seed_doc"] if k in a1.index else v for k, v in zip(key, r1.ci_lo)]
        r1["ci_hi"] = [a1.loc[k, "ci_hi_seed_doc"] if k in a1.index else v for k, v in zip(key, r1.ci_hi)]
    tgt = CFG["turkic_targets"]; dets = [x for x in FT_SHALLOW if x in set(r1.detector) | set(lo.detector)]
    fig, axes = plt.subplots(1, 2, figsize=(FULL, 2.2), sharey=True, gridspec_kw={"wspace": 0.08})
    for ax, df, letter, xl in ((axes[0], r1, "a", "ΔAUROC  multilingual − monolingual"),
                               (axes[1], lo, "b", "ΔAUROC  target held out − multilingual")):
        hairgrid(ax); zero_line(ax)
        for k, det in enumerate(dets):
            g = df[df.detector == det].set_index("target").reindex(tgt); t = DET_TYPE[det]
            yy = np.arange(len(tgt)) + (k - (len(dets) - 1) / 2) * 0.17
            ax.errorbar(g.delta, yy, xerr=[g.delta - g.ci_lo, g.ci_hi - g.delta], fmt=DET_MARK[det], ms=3.8, color=TYPE_COLOR[t],
                        mfc=TYPE_COLOR[t] if det in ("mdeberta", "ngram") else "white", mew=0.9, elinewidth=0.8, capsize=0, zorder=3,
                        label=DET_NAME[det])
        ax.set_xlabel(xl); panel(ax, letter, x=-0.03 if letter == "b" else -0.12)
    axes[0].set_yticks(range(len(tgt)), [f"{l}  {LNAME[l]}" for l in tgt]); axes[0].invert_yaxis()
    axes[0].legend(loc="lower center", bbox_to_anchor=(1.05, 1.1), ncol=4, fontsize=6.5, handletextpad=0.3, columnspacing=1.2)
    out = pd.concat([r1.assign(panel="a"), lo.assign(panel="b")], ignore_index=True)
    save(fig, "fig4_rq1_lolo", out, ["09_statistics/rq1_multi_vs_mono.csv", "09_statistics/rq2_lolo.csv"],
         "Effect of multilingual training (a) and of withholding the target language (b) on Turkic targets.")
run(fig4_rq1_lolo)

## Fig. 5 — Generalisation to an unseen generator family (RQ3)
ΔAUROC = model trained without the family (E3) − model trained with it (E2/all), evaluated on human texts + the held-out family's machine texts. Large marker: pooled over languages (95 % CI); small grey dots: individual languages.

In [ ]:
def fig5_lofo():
    r3 = pd.read_csv(S / "rq3_lofo.csv")
    dets = [x for x in FT_SHALLOW if x in set(r3.detector)]
    fams = [f for f in FAM_ORDER if f in set(r3.heldout_family)]
    fig, axes = plt.subplots(1, len(dets), figsize=(FULL, 2.35), sharey=True, sharex=True, gridspec_kw={"wspace": 0.1})
    for k, (ax, det) in enumerate(zip(np.atleast_1d(axes), dets)):
        g = r3[r3.detector == det]; t = DET_TYPE[det]
        hairgrid(ax); zero_line(ax)
        for i, f in enumerate(fams):
            pl = g[(g.heldout_family == f) & (g.lang != "all")]
            ax.scatter(pl.delta, np.full(len(pl), i) + np.linspace(-0.22, 0.22, len(pl)), s=6, color=AXIS, zorder=2, lw=0)
            a = g[(g.heldout_family == f) & (g.lang == "all")]
            if len(a):
                a = a.iloc[0]
                ax.errorbar(a.delta, i, xerr=[[a.delta - a.ci_lo], [a.ci_hi - a.delta]], fmt=TYPE_MARK[t], ms=4.5, color=TYPE_COLOR[t],
                            mec="white", mew=0.4, elinewidth=1, capsize=0, zorder=3)
        ax.set_title(DET_NAME[det], fontsize=7.5)
        lo_ = np.floor(np.nanmin(r3.delta) * 50) / 50
        ticks = np.round(np.linspace(lo_, 0, 3), 3)
        ax.set_xticks(ticks, [f"{t:.2f}" if t else "0" for t in ticks])
    axes[0].set_yticks(range(len(fams)), [FAM_NAME[f] for f in fams], fontsize=6.3); axes[0].invert_yaxis()
    fig.supxlabel("ΔAUROC  unseen − seen family", fontsize=7.5, y=-0.04)
    save(fig, "fig5_lofo_unseen_family", r3, ["09_statistics/rq3_lofo.csv"],
         "Leave-one-family-out: change in AUROC when the generator family is excluded from training.")
run(fig5_lofo)

## Fig. 6 — Robustness to text transformations (RQ4)
Cell = ΔAUROC (attacked − clean) over all languages; grey text = 95 % CI includes 0. "machine only": transformation applied to machine texts (evasion); "both": applied to human and machine texts (noisy input). Left column: clean AUROC.

In [ ]:
ATT = [("T1_para", "Para-\nphrase", ""), ("T2_bt", "Back-\ntransl.", ""),
       ("T3_h05", "Homoglyph\nmachine only", "5%"), ("T3_h15", "Homoglyph\nmachine only", "15%"),
       ("T3_h05both", "Homoglyph\nboth", "5%"), ("T3_h15both", "Homoglyph\nboth", "15%"),
       ("T4_t05", "Typos\nmachine only", "5%"), ("T4_t15", "Typos\nmachine only", "15%"),
       ("T4_t05both", "Typos\nboth", "5%"), ("T4_t15both", "Typos\nboth", "15%")]

def fig6_robustness():
    r4 = pd.read_csv(S / "rq4_robustness.csv")
    r4 = r4[(r4.lang == "all") & ~r4.detector.str.startswith("DiD")]
    atts = [a for a in ATT if a[0] in set(r4.attack)]
    dets = [x for x in DET_ORDER if x in set(r4.detector)]
    D = r4.pivot(index="detector", columns="attack", values="delta").reindex(index=dets, columns=[a[0] for a in atts])
    lo = r4.pivot(index="detector", columns="attack", values="ci_lo").reindex_like(D)
    hi = r4.pivot(index="detector", columns="attack", values="ci_hi").reindex_like(D)
    clean = r4.groupby("detector").auroc_clean.first().reindex(dets)
    fig = plt.figure(figsize=(FULL, 0.27 * len(dets) + 1.2))
    gs = fig.add_gridspec(1, 2, width_ratios=[1, len(atts)], wspace=0.03)
    axc, ax = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])
    heat(axc, clean.values[:, None], SEQ, 0.5, 1.0, fs=6.2)
    axc.set_xticks([0], ["clean\nAUROC"], fontsize=6.3); axc.xaxis.tick_top()
    axc.set_yticks(range(len(dets)), [DET_NAME[x] for x in dets])
    vmax = max(0.3, np.ceil(np.nanmax(np.abs(D.values)) * 10) / 10)
    im = heat(ax, D.values, DIV.reversed(), -vmax, vmax, fmt="{:+.2f}", fs=6.0, dim=(lo.values <= 0) & (hi.values >= 0))
    ax.set_yticks([]); ax.xaxis.tick_top()
    ax.set_xticks(range(len(atts)), [a[2] for a in atts], fontsize=6.3)
    groups = {}
    for j, a in enumerate(atts):
        groups.setdefault(a[1], []).append(j)
    from matplotlib.transforms import blended_transform_factory
    tr = blended_transform_factory(ax.transData, ax.transAxes)
    for name, js in groups.items():
        ax.text(np.mean(js), 1.075 if len(js) > 1 else 1.02, name, transform=tr, ha="center", va="bottom", fontsize=6.3, color=INK)
        if len(js) > 1:
            ax.plot([js[0] - 0.4, js[-1] + 0.4], [1.065, 1.065], color=AXIS, lw=0.6, clip_on=False, transform=tr)
    for j in range(1, len(atts)):
        if atts[j][1] != atts[j - 1][1]:
            ax.axvline(j - 0.5, color="white", lw=1.5)
    for b in (1.5, 3.5):
        ax.axhline(b, color="white", lw=1.5); axc.axhline(b, color="white", lw=1.5)
    for a_ in (ax, axc):
        a_.set_ylim(len(dets) - 0.5, -0.5)
    cb = fig.colorbar(im, ax=[axc, ax], shrink=0.75, pad=0.015, aspect=25)
    cb.set_label("ΔAUROC  attacked − clean  (red = degradation)", fontsize=6.8); cb.outline.set_visible(False); cb.ax.tick_params(labelsize=6.5, length=2)
    save(fig, "fig6_robustness", r4, ["09_statistics/rq4_robustness.csv"],
         "Change in AUROC under paraphrasing, back-translation, homoglyph and typo transformations.")
run(fig6_robustness)

## Fig. 7 — Homoglyph attack and the Unicode defence (RQ4)
Per detector: clean AUROC (│), after the attack (○) and after NFKC + confusable mapping (●). The defence is applied to every test text, so its cost on clean text is included.

In [ ]:
def fig7_defence():
    d = pd.read_csv(S / "rq4_defence.csv"); d = d[d.lang == "all"]
    tags = [t for t in ("T3_h05", "T3_h15") if t in set(d.attack)]
    dets = [x for x in DET_ORDER if x in set(d.detector)]
    fig, axes = plt.subplots(1, len(tags), figsize=(FULL, 0.22 * len(dets) + 0.9), sharey=True, gridspec_kw={"wspace": 0.08})
    for k, (ax, tag) in enumerate(zip(np.atleast_1d(axes), tags)):
        g = d[d.attack == tag].set_index("detector").reindex(dets)
        hairgrid(ax)
        for i, det in enumerate(dets):
            r_ = g.loc[det]; col = TYPE_COLOR[DET_TYPE[det]]
            ax.annotate("", xy=(r_.auroc_defended, i), xytext=(r_.auroc_attacked, i),
                        arrowprops=dict(arrowstyle="-|>", color=col, lw=0.9, shrinkA=3, shrinkB=3, mutation_scale=6))
            ax.plot(r_.auroc_attacked, i, "o", mfc="white", mec=col, mew=1, ms=4, zorder=3)
            ax.plot(r_.auroc_defended, i, "o", color=col, mec="white", mew=0.4, ms=4.2, zorder=4)
            ax.plot(r_.auroc_clean, i, "|", color=INK, ms=7, mew=1, zorder=5)
        ax.set_title(f"{int(tag[-2:])}% of eligible characters replaced", fontsize=7.3)
        ax.set_xlabel("AUROC"); ax.set_xlim(min(0.2, np.nanmin(g.auroc_attacked) - 0.05), 1.01)
        panel(ax, "ab"[k], x=-0.03 if k else -0.42, y=1.02)
    axes[0].set_yticks(range(len(dets)), [DET_NAME[x] for x in dets]); axes[0].invert_yaxis()
    h = [Line2D([], [], ls="", marker="|", color=INK, ms=7, mew=1, label="clean"),
         Line2D([], [], ls="", marker="o", mfc="white", mec=INK2, ms=4, label="attacked"),
         Line2D([], [], ls="", marker="o", color=INK2, ms=4, label="attacked + defence")]
    fig.legend(handles=h, loc="upper center", ncol=3, bbox_to_anchor=(0.55, 1.05))
    save(fig, "fig7_homoglyph_defence", d, ["09_statistics/rq4_defence.csv"],
         "Homoglyph substitution and recovery with Unicode normalisation + confusable mapping.")
run(fig7_defence)

## Fig. 8 — External test: COLING-2025 Kazakh subset (GPT-4o; different topics, prompts and pipeline)
**a** AUROC on the file as distributed (raw, ○) vs after our normalisation (norm, ●) — a drop after normalisation means the detector used formatting artefacts. **b** Trained detectors on the normalised file under three training regimes (OpenAI family excluded = truly unseen generator family).

In [ ]:
REGIME = {"E1_kk": ("Kazakh only (E1)", "#1baf7a", "^"), "E2_all": ("all languages, all families (E2)", "#2a78d6", "o"),
          "E3_no_openai": ("all languages, OpenAI held out (E3)", "#eb6834", "s")}

def fig8_external():
    e = pd.read_csv(S / "external_coling_kk.csv")
    a = e[e.model.isin(["E2_all", "zero-shot"])]
    dets = [x for x in DET_ORDER if x in set(a.detector)]
    fig, axes = plt.subplots(1, 2, figsize=(FULL, 0.2 * len(dets) + 1.0), gridspec_kw={"width_ratios": [1.15, 1], "wspace": 0.55})
    ax = axes[0]; hairgrid(ax)
    for i, det in enumerate(dets):
        g = a[a.detector == det].set_index("version"); col = TYPE_COLOR[DET_TYPE[det]]
        if {"raw", "norm"} <= set(g.index):
            ax.plot([g.loc["raw", "auroc"], g.loc["norm", "auroc"]], [i, i], color=col, lw=1, alpha=0.6)
            ax.plot(g.loc["raw", "auroc"], i, "o", mfc="white", mec=col, mew=1, ms=4)
            ax.plot(g.loc["norm", "auroc"], i, "o", color=col, mec="white", mew=0.4, ms=4.2)
    ax.set_yticks(range(len(dets)), [DET_NAME[x] for x in dets]); ax.invert_yaxis(); ax.set_xlabel("AUROC on COLING-kk")
    ax.legend(handles=[Line2D([], [], ls="", marker="o", mfc="white", mec=INK2, ms=4, label="raw"),
                       Line2D([], [], ls="", marker="o", color=INK2, ms=4, label="normalised")], loc="lower left",
              bbox_to_anchor=(0.0, 1.0), ncol=2, borderaxespad=0.2)
    panel(ax, "a", x=-0.45, y=1.06)
    ax = axes[1]; hairgrid(ax)
    b = e[(e.version == "norm") & e.model.isin(REGIME)]
    tdets = [x for x in FT_SHALLOW if x in set(b.detector)]
    for k, (m, (lab, col, mk)) in enumerate(REGIME.items()):
        g = b[b.model == m].set_index("detector").reindex(tdets)
        yy = np.arange(len(tdets)) + (k - 1) * 0.22
        ax.errorbar(g.auroc, yy, xerr=[g.auroc - g.ci_lo, g.ci_hi - g.auroc], fmt=mk, ms=3.8, color=col, mec="white", mew=0.4,
                    elinewidth=0.8, capsize=0, label=lab, zorder=3)
    ax.set_yticks(range(len(tdets)), [DET_NAME[x] for x in tdets]); ax.invert_yaxis(); ax.set_xlabel("AUROC on COLING-kk (normalised)")
    ax.legend(loc="lower left", bbox_to_anchor=(-0.05, 1.0), fontsize=6.1, handletextpad=0.3, borderaxespad=0.2, ncol=1)
    panel(ax, "b", x=-0.35, y=1.06)
    save(fig, "fig8_external_coling_kk", e, ["09_statistics/external_coling_kk.csv"],
         "External Kazakh test set (COLING-2025, GPT-4o): artefact sensitivity (a) and training regime (b).")
run(fig8_external)

## Fig. 9 — Operating-point transfer: threshold set at 5 % FPR on the training language
A monolingual detector (E1) is deployed with the threshold that gives 5 % FPR on its **own** validation humans (~90–107 texts). **a/c** realised false-positive rate on each test language (nominal 5 %; colour saturates at the maximum); **b/d** true-positive rate at that threshold. Mean over 3 seeds. The 5 % point is used because a 1 % threshold rests on the single most machine-like validation human; the 1 % version is in `source_data` (and is used automatically if notebook 13 has not been run).

In [ ]:
def fig9_threshold_transfer():
    if (PH / "a4_threshold_transfer.csv").exists():
        tt = pd.read_csv(PH / "a4_threshold_transfer.csv")
        nominal = 0.05 if (tt.nominal_fpr == 0.05).any() else float(tt.nominal_fpr.min())
        out = tt[(tt.nominal_fpr == nominal) & tt.detector.isin(["mdeberta", "xlmr"])].rename(columns={"realised_fpr": "fpr", "realised_tpr": "tpr"})
        src = ["09_statistics/posthoc/a4_threshold_transfer.csv"]
    else:                                                   # fallback: 1 % threshold from notebook 09
        ml = pd.read_csv(R / "metrics_long.csv"); nominal = 0.01; parts = []
        for det in [x for x in ("mdeberta", "xlmr") if x in set(ml.detector)]:
            g = ml[(ml.exp == "E1") & (ml.detector == det) & (ml["eval"] == "test") & (ml.family_subset == "all") & ml.lang.isin(ORDER)]
            parts.append(g.groupby(["train", "lang"])[["fpr_at_src1", "tpr_at_src1"]].mean().reset_index().assign(detector=det))
        out = pd.concat(parts, ignore_index=True).rename(columns={"fpr_at_src1": "fpr", "tpr_at_src1": "tpr"})
        src = ["08_results/metrics_long.csv"]
    dets = [d for d in ("mdeberta", "xlmr") if d in set(out.detector)]
    vmax_fpr = max(2.5 * nominal, out.fpr.max())
    fig, axes = plt.subplots(len(dets), 2, figsize=(FULL, 2.55 * len(dets)), squeeze=False, gridspec_kw={"wspace": 0.35, "hspace": 0.5})
    for r_, det in enumerate(dets):
        g = out[out.detector == det]
        for c, (col, cmap, vmax, title) in enumerate((("fpr", SEQ_ORANGE, vmax_fpr, f"FPR % (nominal {nominal:.0%})"),
                                                       ("tpr", SEQ, 1.0, "TPR %"))):
            ax = axes[r_, c]
            M = g.pivot(index="train", columns="lang", values=col).reindex(index=ORDER, columns=ORDER).values
            im = heat(ax, M * 100, cmap, 0, vmax * 100, fmt="{:.0f}", fs=6.2)
            for i in range(6):
                ax.add_patch(Rectangle((i - 0.5, i - 0.5), 1, 1, fill=False, ec=INK, lw=0.9))
            ax.axhline(3.5, color="white", lw=2); ax.axvline(3.5, color="white", lw=2)
            ax.set_xticks(range(6), ORDER); ax.set_yticks(range(6), ORDER)
            ax.set_xlabel("test language", fontsize=7, color=INK2); ax.set_ylabel("training language", fontsize=7, color=INK2)
            ax.set_title(f"{DET_NAME[det]} · {title}", fontsize=7.3)
            cb = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.03, aspect=18); cb.outline.set_visible(False)
            cb.set_label("%", fontsize=6.5); cb.ax.tick_params(labelsize=6, length=2)
            if c == 0:                                       # mark the nominal level on the colour bar
                cb.ax.axhline(nominal * 100, color=INK, lw=0.8)
            panel(ax, "abcd"[2 * r_ + c], x=-0.14)
    save(fig, "fig9_threshold_transfer", out, src,
         f"Threshold transfer: realised FPR and TPR on each test language at the {nominal:.0%}-FPR validation threshold of the training language.")
run(fig9_threshold_transfer)

## Fig. 10 — Which generators are hard to detect?
AUROC of each detector against human texts + one generator's machine texts, averaged over the six languages (test split).

In [ ]:
def fig10_generators():
    eg = pd.read_csv(R / "error_generator_prompt.csv"); eg = eg[eg.factor == "generator"]
    M = eg.groupby(["detector", "level"]).auroc.mean().unstack()
    dets = [x for x in DET_ORDER if x in M.index]; gens = [g for g in GEN_NAME if g in M.columns]
    M = M.reindex(index=dets, columns=gens)
    fig, ax = plt.subplots(figsize=(HALF + 0.9, 0.24 * len(dets) + 1.0))
    im = heat(ax, M.values, SEQ, 0.4, 1.0, fs=6.2)
    ax.set_xticks(range(len(gens)), [GEN_NAME[g] for g in gens], rotation=30, ha="right", fontsize=6.5)
    ax.set_yticks(range(len(dets)), [DET_NAME[x] for x in dets])
    for b in (1.5, 3.5):
        ax.axhline(b, color="white", lw=1.5)
    cb = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02, aspect=22); cb.outline.set_visible(False)
    cb.set_label("AUROC (mean over languages)", fontsize=6.8); cb.ax.tick_params(labelsize=6, length=2)
    save(fig, "fig10_auroc_by_generator", M.reset_index(), ["08_results/error_generator_prompt.csv"],
         "AUROC per generator (mean over languages).")
run(fig10_generators)

## Fig. 11 — Why the Kazakh-adapted scorer fails for likelihood-based detection (RQ5 mechanism)
**a** AUROC of the mean log-likelihood detector with the generic (Qwen3.5-4B) and the Kazakh-adapted (ISSAI) scoring LM, normalised **per token** (○, as in the pre-registered analysis) or **per character** (●, tokenizer-neutral). The two scorers use different tokenizers, so only the per-character comparison is like-for-like. **b** Likelihood shift: gain = bits/char(generic) − bits/char(Kazakh); plotted is gain(human) − gain(machine) with 95 % document-bootstrap CI. Positive = the Kazakh LM raises the likelihood of human text more than that of machine text, which shrinks the gap a likelihood detector relies on. Exploratory (notebook 13).

In [ ]:
def fig11_rq5_mechanism():
    r5 = pd.read_csv(S / "rq5_language_adapted_scorer.csv"); r5 = r5[r5.method == "loglik"]
    bpc = pd.read_csv(PH / "b_bpc_auroc.csv"); did = pd.read_csv(PH / "b_memorisation_probe.csv")
    rows = [("test", l, f"{l}  {LNAME[l]}") for l in ORDER] + [("ext_coling_norm", "kk", "kk  COLING-kk")]
    SC = {"generic": (TYPE_COLOR["fine-tuned"], "o", -0.14, "generic LM"), "kazakh": (TYPE_COLOR["shallow"], "s", 0.14, "Kazakh-adapted LM")}
    fig, axes = plt.subplots(1, 2, figsize=(FULL, 2.6), sharey=True, gridspec_kw={"width_ratios": [1.25, 1], "wspace": 0.08})
    ax = axes[0]; hairgrid(ax); recs = []
    for i, (ev, l, _) in enumerate(rows):
        for sc, (col, mk, dy, _) in SC.items():
            tok = r5[(r5["eval"] == ev) & (r5.lang == l)][f"auroc_{sc}"]
            ch = bpc[(bpc["eval"] == ev) & (bpc.lang == l) & (bpc.scorer == sc)].auroc_per_char_loglik
            if tok.empty or ch.empty:
                continue
            a_t, a_c = float(tok.iloc[0]), float(ch.iloc[0])
            ax.annotate("", xy=(a_c, i + dy), xytext=(a_t, i + dy),
                        arrowprops=dict(arrowstyle="-|>", color=col, lw=0.8, shrinkA=2.5, shrinkB=2.5, mutation_scale=5))
            ax.plot(a_t, i + dy, mk, mfc="white", mec=col, mew=1, ms=4, zorder=3)
            ax.plot(a_c, i + dy, mk, color=col, mec="white", mew=0.4, ms=4.4, zorder=4)
            recs.append({"panel": "a", "eval": ev, "lang": l, "scorer": sc, "auroc_per_token": a_t, "auroc_per_char": a_c})
    ax.set_yticks(range(len(rows)), [r[2] for r in rows]); ax.invert_yaxis()
    ax.set_xlabel("AUROC of the log-likelihood detector")
    h = [Line2D([], [], ls="", marker=v[1], color=v[0], mec="white", ms=5, label=v[3]) for v in SC.values()] + \
        [Line2D([], [], ls="", marker="o", mfc="white", mec=INK2, ms=4, label="per token"),
         Line2D([], [], ls="", marker="o", color=INK2, ms=4, label="per character")]
    ax.legend(handles=h, loc="lower left", bbox_to_anchor=(0.0, 1.0), ncol=4, fontsize=6.2, handletextpad=0.2, columnspacing=0.9, borderaxespad=0.2)
    panel(ax, "a", x=-0.24, y=1.09)
    ax = axes[1]; hairgrid(ax); zero_line(ax)
    for i, (ev, l, _) in enumerate(rows):
        g = did[(did["eval"] == ev) & (did.lang == l)]
        if g.empty:
            continue
        r_ = g.iloc[0]; sig = r_.did_ci_lo > 0 or r_.did_ci_hi < 0
        ax.errorbar(r_.did_human_minus_machine, i, xerr=[[r_.did_human_minus_machine - r_.did_ci_lo], [r_.did_ci_hi - r_.did_human_minus_machine]],
                    fmt="s", ms=4, color=TYPE_COLOR["shallow"] if sig else MUTED, mfc=TYPE_COLOR["shallow"] if sig else "white",
                    mec=TYPE_COLOR["shallow"] if sig else MUTED, mew=0.9, elinewidth=0.9, capsize=0, zorder=3)
        recs.append({"panel": "b", "eval": ev, "lang": l, "did": r_.did_human_minus_machine, "ci_lo": r_.did_ci_lo, "ci_hi": r_.did_ci_hi})
    ax.set_xlabel("gain(human) − gain(machine)  [bits / char]")
    ax.text(1.0, 1.01, "hollow = 95% CI includes 0", transform=ax.transAxes, ha="right", va="bottom", fontsize=6, color=MUTED)
    panel(ax, "b", x=-0.03, y=1.09)
    save(fig, "fig11_rq5_mechanism", pd.DataFrame(recs),
         ["09_statistics/rq5_language_adapted_scorer.csv", "09_statistics/posthoc/b_bpc_auroc.csv", "09_statistics/posthoc/b_memorisation_probe.csv"],
         "Kazakh-adapted vs generic scoring LM: per-token vs per-character log-likelihood AUROC (a) and the human-vs-machine likelihood shift (b).")
run(fig11_rq5_mechanism)

## Fig. 12 — Llama-3.3-70B: an unseen-family penalty or an atypical generator? (RQ3)
**a** Relative error ratio RER = (1 − AUROC_unseen) / (1 − AUROC_seen) when a generator family is left out of training (seed-matched E3 vs E2/all, pooled over languages; 95 % document-bootstrap CI; log scale). RER = 1: no unseen-family penalty relative to that family's own difficulty. **b** Surface profile of each generator's texts: standardised difference from human texts of the same language (mean over languages; blue = lower than human, red = higher), and the cross-validated AUROC with which these descriptors separate the generator from the other four (bottom row). Exploratory (notebook 13).

In [ ]:
FEAT_NAME = {"n_words": "length (words)", "mean_sent_words": "sentence length", "mattr50": "lexical diversity (MATTR)", "distinct3": "distinct 3-grams",
             "punct_per_word": "punctuation / word", "comma_per_word": "commas / word", "digit_share": "digit share"}
GEN_SHORT = {"gpt41mini": "GPT-4.1-mini", "haiku45": "Haiku 4.5", "gem25lite": "Gemini 2.5 FL", "llama33_70b": "Llama-3.3", "dsv3": "DeepSeek-V3"}

def fig12_llama():
    c1 = pd.read_csv(PH / "c_lofo_relative_error.csv")
    cd = pd.read_csv(PH / "c_generator_descriptors.csv").set_index("generator")
    ca = pd.read_csv(PH / "c_generator_atypicality.csv").set_index("generator")
    dets = [x for x in FT_SHALLOW if x in set(c1.detector)]; fams = [f for f in FAM_ORDER if f in set(c1.heldout_family)]
    gens = [g for g in GEN_NAME if g in cd.index]; feats = [f for f in FEAT_NAME if f"{f}_z_vs_human" in cd.columns]
    fig = plt.figure(figsize=(FULL, 3.0))
    outer = fig.add_gridspec(1, 2, width_ratios=[1, 1.05], wspace=0.62)
    ax = fig.add_subplot(outer[0]); hairgrid(ax)
    ax.axvline(1, color=INK2, lw=0.6, zorder=1)
    for k, det in enumerate(dets):
        g = c1[c1.detector == det].set_index("heldout_family").reindex(fams); t = DET_TYPE[det]
        yy = np.arange(len(fams)) + (k - (len(dets) - 1) / 2) * 0.17
        hi = g.rer_ci_hi.clip(upper=30)
        ax.errorbar(g.rer, yy, xerr=[g.rer - g.rer_ci_lo, hi - g.rer], fmt=DET_MARK[det], ms=3.8, color=TYPE_COLOR[t],
                    mfc=TYPE_COLOR[t] if det in ("mdeberta", "ngram") else "white", mew=0.9, elinewidth=0.8, capsize=0, zorder=3, label=DET_NAME[det])
    ax.set_xscale("log"); ax.set_xticks([0.2, 0.5, 1, 2, 5, 10, 20], ["0.2", "0.5", "1", "2", "5", "10", "20"]); ax.minorticks_off()
    ax.set_xlim(0.1, 32)
    ax.set_yticks(range(len(fams)), [FAM_NAME[f] for f in fams], fontsize=6.3); ax.invert_yaxis()
    ax.set_xlabel("relative error ratio, unseen / seen (log)")
    ax.legend(loc="lower left", bbox_to_anchor=(-0.02, 1.0), ncol=2, fontsize=6, handletextpad=0.2, columnspacing=0.8, borderaxespad=0.2)
    panel(ax, "a", x=-0.42, y=1.17)
    inner = outer[1].subgridspec(2, 1, height_ratios=[len(feats), 1.15], hspace=0.08)
    Z = cd.loc[gens, [f"{f}_z_vs_human" for f in feats]].values.astype(float).T          # features x generators
    vmax = float(np.ceil(np.nanmax(np.abs(Z)) * 2) / 2)
    axh = fig.add_subplot(inner[0]); im = heat(axh, Z, DIV, -vmax, vmax, fmt="{:+.2f}", fs=5.8)
    axh.set_yticks(range(len(feats)), [FEAT_NAME[f] for f in feats], fontsize=6.2)
    axh.set_xticks(range(len(gens)), [GEN_SHORT.get(g, GEN_NAME[g]) for g in gens], fontsize=6, rotation=30, ha="left", rotation_mode="anchor"); axh.xaxis.tick_top()
    panel(axh, "b", x=-0.62, y=1.22)
    axa = fig.add_subplot(inner[1]); A = ca.reindex(gens).cv_auroc_vs_other_generators.values[None, :]
    heat(axa, A, SEQ, 0.5, 1.0, fmt="{:.2f}", fs=5.8)
    axa.set_yticks([0], ["atypicality (CV-AUROC)"], fontsize=6.2); axa.set_xticks([])
    cb = fig.colorbar(im, ax=[axh, axa], shrink=0.85, pad=0.03, aspect=24); cb.outline.set_visible(False)
    cb.set_label("standardised difference vs human", fontsize=6.3); cb.ax.tick_params(labelsize=6, length=2)
    out = pd.concat([c1.assign(panel="a"), cd.loc[gens].reset_index().assign(panel="b"), ca.reset_index().assign(panel="b_atypicality")], ignore_index=True)
    save(fig, "fig12_llama_unseen_or_atypical", out,
         ["09_statistics/posthoc/c_lofo_relative_error.csv", "09_statistics/posthoc/c_generator_descriptors.csv", "09_statistics/posthoc/c_generator_atypicality.csv"],
         "Relative unseen-family error (a) and surface profile / atypicality of each generator's texts (b).")
run(fig12_llama)

## Fig. 13 — Does fusing detectors help? (exploratory)
Rank-average fusion: each member's score is mapped to its empirical CDF on the pooled validation split (no test data), then averaged. Plotted: ensemble − mDeBERTa alone, pooled over languages, **a** ΔAUROC (95 % paired document-bootstrap CI) and **b** ΔTPR at 5 % FPR. Adding the character n-gram model buys robustness to strong character-level attacks; adding Binoculars hurts everywhere.

In [ ]:
EVAL_NAME = {"test": "clean test", "T1_para": "paraphrase", "T2_bt": "back-translation", "T3_h05": "homoglyph 5%", "T3_h15": "homoglyph 15%",
             "T4_t05": "typos 5%", "T4_t15": "typos 15%", "T3_h05both": "homoglyph 5% (both)", "T3_h15both": "homoglyph 15% (both)",
             "T4_t05both": "typos 5% (both)", "T4_t15both": "typos 15% (both)", "ext_coling_norm": "COLING-kk"}
COMBO_STYLE = {"mDeBERTa+ngram": ("+ char n-gram", TYPE_COLOR["shallow"], "s"), "mDeBERTa+Binoculars": ("+ Binoculars", TYPE_COLOR["zero-shot"], "^"),
               "mDeBERTa+XLM-R+ngram+Binoculars": ("+ XLM-R + n-gram + Binoculars", INK2, "D")}

def fig13_fusion():
    d = pd.read_csv(PH / "d_ensemble.csv"); d = d[d.lang == "all"]
    evs = [e for e in EVAL_NAME if e in set(d["eval"])]; combos = [c for c in COMBO_STYLE if c in set(d.ensemble)]
    fig, axes = plt.subplots(1, 2, figsize=(FULL, 0.2 * len(evs) + 1.0), sharey=True, gridspec_kw={"width_ratios": [1.3, 1], "wspace": 0.08})
    for ax, letter in zip(axes, "ab"):
        hairgrid(ax); zero_line(ax)
        for b in (0.5, 6.5, 10.5):
            if b < len(evs) - 0.5:
                ax.axhline(b, color=GRID, lw=0.6)
        for k, c in enumerate(combos):
            lab, col, mk = COMBO_STYLE[c]; g = d[d.ensemble == c].set_index("eval").reindex(evs)
            yy = np.arange(len(evs)) + (k - (len(combos) - 1) / 2) * 0.22
            if letter == "a":
                ax.errorbar(g.delta, yy, xerr=[g.delta - g.ci_lo, g.ci_hi - g.delta], fmt=mk, ms=3.8, color=col, mec="white", mew=0.4,
                            elinewidth=0.8, capsize=0, zorder=3, label=lab)
            else:
                ax.plot(g.tpr5_ensemble - g.tpr5_best_member, yy, mk, ms=3.8, color=col, mec="white", mew=0.4, zorder=3)
        ax.set_xlabel("ΔAUROC  ensemble − mDeBERTa" if letter == "a" else "ΔTPR at 5% FPR")
        panel(ax, letter, x=-0.32 if letter == "a" else -0.03, y=1.06)
    axes[0].set_yticks(range(len(evs)), [EVAL_NAME[e] for e in evs], fontsize=6.3); axes[0].invert_yaxis()
    axes[0].legend(loc="lower left", bbox_to_anchor=(0.0, 1.0), ncol=3, fontsize=6.2, handletextpad=0.2, columnspacing=0.9, borderaxespad=0.2)
    save(fig, "fig13_fusion", d, ["09_statistics/posthoc/d_ensemble.csv"],
         "Rank-average fusion with mDeBERTa: change in AUROC (a) and TPR@5%FPR (b) on clean, attacked and external data.")
run(fig13_fusion)

## Appendix figures

In [ ]:
def figA1_rq5():
    r5 = pd.read_csv(S / "rq5_language_adapted_scorer.csv")
    meths = [m for m in ("binoculars", "fastdetect", "loglik", "logrank") if m in set(r5.method)]
    MN = {"binoculars": "Binoculars", "fastdetect": "Fast-DetectGPT", "loglik": "Log-likelihood", "logrank": "Log-rank"}
    rows = [("test", l, f"{l}  {LNAME[l]}") for l in ORDER] + [("ext_coling_norm", "kk", "kk  COLING-kk")]
    fig, axes = plt.subplots(1, len(meths), figsize=(FULL, 2.3), sharey=True, sharex=True, gridspec_kw={"wspace": 0.1})
    for ax, m in zip(np.atleast_1d(axes), meths):
        hairgrid(ax); zero_line(ax)
        g = r5[r5.method == m]
        for i, (ev, l, _) in enumerate(rows):
            r_ = g[(g["eval"] == ev) & (g.lang == l)]
            if len(r_):
                r_ = r_.iloc[0]; col = TYPE_COLOR["zero-shot"] if l == "kk" else MUTED
                ax.errorbar(r_.delta, i, xerr=[[r_.delta - r_.ci_lo], [r_.ci_hi - r_.delta]], fmt="^", ms=4, color=col,
                            mec="white", mew=0.4, elinewidth=0.9, capsize=0, zorder=3)
        ax.set_title(MN[m], fontsize=7.5)
    axes[0].set_yticks(range(len(rows)), [r[2] for r in rows]); axes[0].invert_yaxis()
    fig.supxlabel("ΔAUROC  Kazakh-adapted − generic scoring LM", fontsize=7.5, y=-0.04)
    save(fig, "figA1_rq5_language_adapted_scorer", r5, ["09_statistics/rq5_language_adapted_scorer.csv"],
         "Kazakh-adapted vs generic scoring model for zero-shot detectors (Kazakh highlighted).")
run(figA1_rq5)

def figA2_length():
    el = pd.read_csv(R / "error_length.csv"); el = el[el.lang != "all"]
    M = el.groupby(["detector", "len_bin"]).auroc.mean().unstack().reindex(columns=["short", "mid", "long"])
    dets = [x for x in DET_ORDER if x in M.index]; M = M.reindex(dets)
    fig, ax = plt.subplots(figsize=(HALF, 0.22 * len(dets) + 0.8))
    im = heat(ax, M.values, SEQ, 0.5, 1.0, fs=6.2)
    ax.set_xticks(range(3), ["short", "mid", "long"]); ax.xaxis.tick_top()
    ax.set_yticks(range(len(dets)), [DET_NAME[x] for x in dets])
    cb = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.03, aspect=20); cb.outline.set_visible(False)
    cb.set_label("AUROC within length tertile", fontsize=6.5); cb.ax.tick_params(labelsize=6, length=2)
    save(fig, "figA2_length_tertiles", M.reset_index(), ["08_results/error_length.csv"], "AUROC within text-length tertiles (mean over languages).")
run(figA2_length)

def figA3_overlap():
    J = pd.read_csv(R / "error_overlap_jaccard.csv", index_col=0)
    dets = [x for x in DET_ORDER if x in J.index]; J = J.reindex(index=dets, columns=dets)
    M = J.values.copy(); M[np.triu_indices_from(M)] = np.nan
    fig, ax = plt.subplots(figsize=(HALF + 0.6, HALF))
    im = heat(ax, M, SEQ, 0, 1, fs=5.8)
    ax.set_xticks(range(len(dets)), [DET_NAME[x] for x in dets], rotation=45, ha="right", fontsize=6)
    ax.set_yticks(range(len(dets)), [DET_NAME[x] for x in dets], fontsize=6)
    cb = fig.colorbar(im, ax=ax, shrink=0.7, pad=0.02); cb.outline.set_visible(False)
    cb.set_label("Jaccard overlap of errors", fontsize=6.5); cb.ax.tick_params(labelsize=6, length=2)
    save(fig, "figA3_error_overlap", J.reset_index(), ["08_results/error_overlap_jaccard.csv"],
         "Overlap of misclassified test documents between detectors (low = complementary).")
run(figA3_overlap)

def figA4_audit():
    a = pd.read_csv(R / "data_audit.csv")
    SETN = {f"ours_{l}": f"{l}  (ours)" for l in ORDER} | {"ext_coling_raw": "COLING-kk raw", "ext_coling_norm": "COLING-kk norm"}
    sets = [s for s in SETN if s in set(a["set"])]; a = a.set_index("set").reindex(sets)
    FEAT = [("auroc_artefacts_only", "formatting artefacts"), ("auroc_length_only", "length"),
            ("auroc_title_start", "starts with title"), ("auroc_stylometric", "stylometric\n(punctuation + sentence length)")]
    FEAT = [f for f in FEAT if f[0] in a.columns]
    fig, axes = plt.subplots(1, len(FEAT), figsize=(FULL, 0.2 * len(sets) + 1.0), sharey=True, sharex=True, gridspec_kw={"wspace": 0.1})
    for ax, (c, lab) in zip(np.atleast_1d(axes), FEAT):
        hairgrid(ax); ax.axvline(0.5, color=INK2, lw=0.6)
        for i, s in enumerate(sets):
            v = a.loc[s, c]
            if pd.notna(v):
                col = TYPE_COLOR["shallow"] if s.startswith("ext") else TYPE_COLOR["fine-tuned"]
                ax.plot([0.5, v], [i, i], color=col, lw=1, alpha=0.5); ax.plot(v, i, "o", color=col, mec="white", mew=0.4, ms=4)
        ax.set_title(lab, fontsize=7); ax.set_xlim(0.4, 1.0); ax.set_xticks([0.5, 0.75, 1.0], ["0.5", "0.75", "1"])
    axes[0].set_yticks(range(len(sets)), [SETN[s] for s in sets]); axes[0].invert_yaxis()
    fig.supxlabel("cross-validated AUROC of a shallow classifier (0.5 = no shortcut)", fontsize=7.5, y=-0.03)
    save(fig, "figA4_dataset_shortcut_audit", a.reset_index(), ["08_results/data_audit.csv"],
         "Dataset audit: how well surface features alone separate human from machine text.")
run(figA4_audit)

## LaTeX tables (booktabs; best value per column in bold; 95 % CI in brackets)

In [ ]:
def latex_table(df, caption, label, col_fmt=None, note=None):
    cols = list(df.columns)
    col_fmt = col_fmt or "l" + "c" * (len(cols) - 1)
    lines = ["\\begin{table}[t]", "\\centering", "\\small", f"\\caption{{{caption}}}", f"\\label{{{label}}}",
             f"\\begin{{tabular}}{{{col_fmt}}}", "\\toprule", " & ".join(str(c) for c in cols) + " \\\\", "\\midrule"]
    for _, r_ in df.iterrows():
        lines.append(" & ".join("" if pd.isna(v) else str(v) for v in r_.values) + " \\\\")
    lines += ["\\bottomrule", "\\end{tabular}"]
    if note:
        lines.append(f"\\\\[2pt]\\parbox{{\\linewidth}}{{\\footnotesize {note}}}")
    lines.append("\\end{table}")
    return "\n".join(lines)

def cell(v, lo, hi, best=False):
    s = f"{v:.3f}"
    s = f"\\textbf{{{s}}}" if best else s
    return f"{s}\\,{{\\scriptsize[{lo:.3f}, {hi:.3f}]}}"

def tables():
    # T1 main AUROC
    d = MAIN[MAIN.lang.isin(ORDER + ["all"])]
    cols = ORDER + ["all"]; dets = [x for x in DET_ORDER if x in set(d.detector)]
    best = d.groupby("lang").auroc.max()
    rows = []
    for det in dets:
        g = d[d.detector == det].set_index("lang")
        rows.append([DET_NAME[det], DET_TYPE[det]] + [cell(g.loc[l, "auroc"], g.loc[l, "ci_lo"], g.loc[l, "ci_hi"], np.isclose(g.loc[l, "auroc"], best[l]))
                                                      if l in g.index else "" for l in cols])
    t1 = pd.DataFrame(rows, columns=["Detector", "Type"] + cols)
    (T / "tab1_main_auroc.tex").write_text(latex_table(t1, "Test AUROC [95\\% CI] per language. Fine-tuned and shallow detectors: multilingual training (E2/all, mean of seeds); zero-shot: no training.", "tab:main", "ll" + "c" * len(cols)))
    t1.to_csv(T / "tab1_main_auroc.csv", index=False)
    # T2 low-FPR regime: TPR at 1 % / 5 % FPR (oracle threshold on the test language); CIs from notebook 13 when available
    if (PH / "a4_lowfpr_main.csv").exists():
        lf = pd.read_csv(PH / "a4_lowfpr_main.csv")
        def ci(det, lang, fpr):
            r_ = lf[(lf.detector == det) & (lf.lang == lang) & np.isclose(lf.fpr, fpr)]
            return "" if r_.empty else f"{r_.tpr.iloc[0]:.2f}\\,{{\\scriptsize[{r_.ci_lo.iloc[0]:.2f}, {r_.ci_hi.iloc[0]:.2f}]}}"
        def pt(det, lang, fpr):
            r_ = lf[(lf.detector == det) & (lf.lang == lang) & np.isclose(lf.fpr, fpr)]
            return "" if r_.empty else f"{r_.tpr.iloc[0]:.2f}"
        d2 = [x for x in DET_ORDER if x in set(lf.detector)]
        t2 = pd.DataFrame([[DET_NAME[x], ci(x, "all", 0.01), ci(x, "all", 0.05)] + [pt(x, l, 0.05) for l in ORDER] for x in d2],
                          columns=["Detector", "TPR@1\\% (all)", "TPR@5\\% (all)"] + [f"{l} @5\\%" for l in ORDER])
        (T / "tab2_tpr_at_low_fpr.tex").write_text(latex_table(t2, "True-positive rate at 1\\% and 5\\% false-positive rate (test split; oracle threshold per evaluation set; mean of seeds) with 95\\% bootstrap CI (threshold re-estimated in every resample).", "tab:lowfpr",
            note="With $n\\approx185$--$211$ human test texts per language the FPR resolution is $\\approx0.5$\\%, so per-language TPR@1\\%FPR is unstable and only the pooled value ($n=1159$) is reported."))
        lf.to_csv(T / "tab2_tpr_at_low_fpr.csv", index=False)
    else:
        ml = pd.read_csv(R / "metrics_long.csv")
        m = ml[(ml["eval"] == "test") & (ml.family_subset == "all") & ml.lang.isin(ORDER + ["all"]) &
               (((ml.exp == "E2") & (ml.train == "all") & (ml.heldout == "none")) | (ml.exp == "E5"))]
        tp = m.groupby(["detector", "lang"]).tpr_at_1fpr_oracle.mean().unstack().reindex(index=dets, columns=cols)
        t2 = tp.map(lambda v: f"{v:.2f}" if pd.notna(v) else "").reset_index()
        t2.insert(0, "Detector", t2.pop("detector").map(DET_NAME))
        (T / "tab2_tpr_at_1fpr.tex").write_text(latex_table(t2, "True-positive rate at 1\\% false-positive rate (test split; mean of seeds).", "tab:tpr1"))
        tp.reset_index().to_csv(T / "tab2_tpr_at_1fpr.csv", index=False)
    # T3 RQ2 coefficients
    cf = pd.read_csv(S / "rq2_family_vs_script.csv")
    t3 = pd.DataFrame({"Detector": cf.detector.map(DET_NAME),
                       "same family": [f"{a:+.3f} [{b:+.3f}, {c:+.3f}]" for a, b, c in zip(cf.coef_same_family, cf.ci_lo_family, cf.ci_hi_family)],
                       "perm. $p$ (family)": cf.perm_p_family.map(lambda v: f"{v:.3f}"),
                       "same script": [f"{a:+.3f} [{b:+.3f}, {c:+.3f}]" for a, b, c in zip(cf.coef_same_script, cf.ci_lo_script, cf.ci_hi_script)],
                       "perm. $p$ (script)": cf.perm_p_script.map(lambda v: f"{v:.3f}")})
    (T / "tab3_family_vs_script.tex").write_text(latex_table(t3, "Fixed-effects regression of logit transfer AUROC on language relation.", "tab:famscript",
        note="Exact permutation over language labels; with six languages the smallest attainable $p$ is 0.067 (family) and 0.10 (script)."))
    # T4 LOFO pooled
    r3 = pd.read_csv(S / "rq3_lofo.csv"); r3 = r3[r3.lang == "all"]
    t4 = r3.pivot(index="detector", columns="heldout_family", values="delta").reindex(index=[x for x in FT_SHALLOW if x in set(r3.detector)],
                                                                                         columns=[f for f in FAM_ORDER if f in set(r3.heldout_family)])
    t4 = t4.map(lambda v: f"{v:+.3f}").reset_index(); t4.insert(0, "Detector", t4.pop("detector").map(DET_NAME))
    (T / "tab4_lofo.tex").write_text(latex_table(t4, "$\\Delta$AUROC when the generator family is unseen in training (pooled over languages).", "tab:lofo"))
    # T5 robustness
    r4 = pd.read_csv(S / "rq4_robustness.csv"); r4 = r4[(r4.lang == "all") & ~r4.detector.str.startswith("DiD")]
    t5 = r4.pivot(index="detector", columns="attack", values="delta").reindex(index=[x for x in DET_ORDER if x in set(r4.detector)],
                                                                             columns=[a[0] for a in ATT if a[0] in set(r4.attack)])
    t5 = t5.map(lambda v: f"{v:+.3f}").reset_index(); t5.insert(0, "Detector", t5.pop("detector").map(DET_NAME))
    (T / "tab5_robustness.tex").write_text(latex_table(t5, "$\\Delta$AUROC (attacked $-$ clean), all languages.", "tab:robust"))
    # T6 external
    e = pd.read_csv(S / "external_coling_kk.csv")
    e["cell"] = [cell(a, b, c) for a, b, c in zip(e.auroc, e.ci_lo, e.ci_hi)]
    t6 = e.pivot_table(index=["detector", "model"], columns="version", values="cell", aggfunc="first").reset_index()
    t6["detector"] = t6.detector.map(DET_NAME)
    (T / "tab6_external_coling_kk.tex").write_text(latex_table(t6, "External test on the COLING-2025 Kazakh subset (GPT-4o): raw vs.\\ normalised formatting.", "tab:external"))
    # T7 RQ1 with seed + document uncertainty (notebook 13)
    if (PH / "a1_rq1_seedaware.csv").exists():
        a1 = pd.read_csv(PH / "a1_rq1_seedaware.csv")
        t7 = pd.DataFrame({"Detector": a1.detector.map(DET_NAME), "Target": a1.target,
                           "$\\Delta$AUROC": a1.delta.map(lambda v: f"{v:+.3f}"),
                           "CI (documents)": [f"[{a:+.3f}, {b:+.3f}]" for a, b in zip(a1.ci_lo_doc, a1.ci_hi_doc)],
                           "CI (seeds + documents)": [f"[{a:+.3f}, {b:+.3f}]" for a, b in zip(a1.ci_lo_seed_doc, a1.ci_hi_seed_doc)],
                           "$p$": a1.p_seed_doc.map(lambda v: f"{v:.3f}"), "$p_{\\mathrm{Holm}}$": a1.p_holm_seed_doc.map(lambda v: f"{v:.3f}"),
                           "seed pairs $>0$": [f"{a}/{b}" for a, b in zip(a1.cross_seed_pos, a1.cross_seed_n)]})
        (T / "tab7_rq1_seedaware.tex").write_text(latex_table(t7, "RQ1 robustness check: multilingual (E2/all) minus monolingual (E1) AUROC with document-only and hierarchical seed + document bootstrap CIs; Holm correction over all 16 contrasts; sign of all cross-seed contrasts.", "tab:rq1seed",
            "llcccccc", note="Exploratory (notebook 13). E2/all also uses about six times more training documents than E1, so the contrast does not isolate language diversity."))
    # T8 RQ2 with seed + document uncertainty and leave-one-pair-out
    if (PH / "a2_rq2_seedaware.csv").exists():
        a2 = pd.read_csv(PH / "a2_rq2_seedaware.csv")
        lp = pd.read_csv(PH / "a2_rq2_leave_pair_out.csv") if (PH / "a2_rq2_leave_pair_out.csv").exists() else None
        rows = []
        for r_ in a2.itertuples():
            row = {"Detector": DET_NAME[r_.detector], "seeds": r_.n_seeds,
                   "same script": f"{r_.coef_script:+.2f} [{r_.ci_seed_doc_script_lo:+.2f}, {r_.ci_seed_doc_script_hi:+.2f}]",
                   "same family": f"{r_.coef_family:+.2f} [{r_.ci_seed_doc_family_lo:+.2f}, {r_.ci_seed_doc_family_hi:+.2f}]",
                   "$P(\\beta_{fam}\\le\\beta_{scr})$": f"{r_.p_family_le_script_seed_doc:.3f}"}
            if lp is not None:
                g = lp[lp.detector == r_.detector]
                row["family $>$ script (subsets)"] = f"{int(g.family_gt_script.sum())}/{len(g)}"
                row["family range"] = f"[{g.coef_family.min():+.2f}, {g.coef_family.max():+.2f}]"
                row["script range"] = f"[{g.coef_script.min():+.2f}, {g.coef_script.max():+.2f}]"
            rows.append(row)
        t8 = pd.DataFrame(rows)
        (T / "tab8_rq2_seedaware.tex").write_text(latex_table(t8, "RQ2 robustness check: family and script coefficients with seed + document bootstrap 95\\% CIs, and their range when any one language pair (15 subsets) or any one language (6 subsets) is removed.", "tab:rq2seed",
            note="Exploratory (notebook 13). Deterministic detectors (n-gram, stylometric) have one seed, so their CIs are document-only."))
    log.info(f"tables written to {T}")
run(tables)

In [ ]:
pd.DataFrame(MANIFEST).to_csv(F / "figure_manifest.csv", index=False)
pd.DataFrame(MANIFEST)